In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.nn as nn

from src import (
    load_experiment, get_transform, get_loaders_from_config,
    collect_layer_inputs, collect_layer_inputs_generic,
    bft, evaluate,
    extract_tree_nodes, compute_node_activations, plot_factor_tree,
    extract_factor_fingerprint, extract_fingerprint_matrix,
    compute_stimulus_similarity, project_stimuli_onto_tree,
    extract_factor_tree_nodes, compute_factor_activations,
    nodes_at_layer, top_stimuli_factor_activations,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
matplotlib.rcParams.update({"figure.dpi": 80})


# Notebook 07: Stimulus-Conditioned BFT Factor Analysis — Digit MLP 40-20

Same analysis pipeline as Notebook 06, applied to the **10-class digit classifier**
(SimpleMLP 784→40→20→10, trained on all MNIST digits 0–9).

**Key differences from Notebook 06:**
- Identity label transform: classes are the 10 digits directly (no even/odd collapsing).
- All 10 MNIST digits appear in training — no excluded digits to use as near-OOD.
- **Real OOD = Fashion-MNIST**: same 28×28 grayscale format, completely different visual domain.

**Section 2** — Factor tree visualisation per output/input factor.

**Section 3** — Factor fingerprints & pairwise stimulus similarity.

**Section 4** — ID sanity check: train vs. test (same digit distribution).

**Section 5** — Real OOD: Fashion-MNIST items projected onto fixed BFT factors.

**Section 5b** — Far OOD: synthetic images (noise, gray, checkerboard, inverted).

## Section 1: Setup — load model, data, run BFT

In [ ]:
EXPERIMENT_DIR = '../experiments/mnist_digit_mlp_40_20_seed0'
# Forward order (L0=784→40, L1=40→20, L2=20→10)
N_BRANCHES     = [1, 2, 5]    # 5 branches at output layer, 2 at middle, leaf at input
K_MAX          = [10, 6, 10]  # up to 10 factors at output (one per class)
STIM_THRESHOLD = 0.7
AUTO_THRESHOLD = 0.70
TOP_N_FACTOR   = 50

model, cfg = load_experiment(EXPERIMENT_DIR, device)
train_loader, test_loader = get_loaders_from_config(cfg)
label_transform = get_transform(cfg['label_transform'])  # None for 'identity'

CLASS_NAMES = {i: str(i) for i in range(cfg['arch_kwargs']['output_dim'])}
IMAGE_SIDE  = cfg['input_side']

# Fashion-MNIST class names (the real OOD domain)
FMNIST_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot',
]

_, test_acc = evaluate(model, test_loader,
                       criterion=nn.CrossEntropyLoss(),
                       label_transform=label_transform, device=device)
print(f'Test accuracy : {test_acc:.4f}')
print(f'Classes       : {CLASS_NAMES}')
print(f'Image side    : {IMAGE_SIDE}')
print(f'OOD domain    : Fashion-MNIST (28×28 grayscale, 10 fashion categories)')
_nb_dir   = os.path.dirname(os.path.abspath('__file__'))
_repo_dir = os.path.dirname(_nb_dir)
FIG_DIR   = os.path.join(_repo_dir, 'figs', '07_digit_stimulus_analysis')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'FIG_DIR: {FIG_DIR}')


In [ ]:
data_test = collect_layer_inputs(
    model, test_loader.dataset,
    label_transform=label_transform, n_per_class=None, device=device,
)
idx = np.random.choice(len(data_test['images']), 1000, replace=False)
all_images   = data_test['images'][idx]       # (N, 1, 28, 28)
all_targets  = data_test['targets'][idx]       # (N,) digit 0–9
all_digits   = data_test['digits'][idx]        # same as all_targets for identity transform
layer_inputs = [layer[idx] for layer in data_test['layer_inputs']]  # list[(N, n_in)] forward order
n_samples    = len(all_images)
print(f'Test samples : {n_samples}')
print(f'Class dist   : {dict(zip(*np.unique(all_targets, return_counts=True)))}')

In [ ]:
tree_root = bft(
    model, layer_inputs,
    k_max=K_MAX, n_branches=N_BRANCHES,
    threshold=AUTO_THRESHOLD,
    stimulus_threshold=STIM_THRESHOLD,
    weighting='img_selectivity', verbose=1,
)

tree_nodes   = extract_tree_nodes(tree_root)
factor_nodes = extract_factor_tree_nodes(tree_root)

max_layer_idx = max(e['layer_idx'] for e in tree_nodes)
root_bft_node = tree_root
l0_nodes      = nodes_at_layer(tree_root, 0)

print(f'\nTree: {len(tree_nodes)} path nodes,  {len(factor_nodes)} factor nodes,  max_layer_idx={max_layer_idx}')
print(f'Root factors (output layer): K={len(root_bft_node["lambdas"])}')
print(f'L0 leaf nodes: {len(l0_nodes)}')
for ln in l0_nodes:
    print(f'  path={ln["path"]}  K={len(ln["lambdas"])}')

## Section 2: Factor Tree Visualisation — one tree per factor

- **2a** — Output (last-layer / root) factors: each corresponds to a group of digit classes.
- **2b** — Input (L0 / leaf) factors, one subplot per leaf node.

In [ ]:
# ── 2a: Output (last-layer / root) factors ────────────────────────────────────
K_root = len(root_bft_node['lambdas'])
print(f'Plotting {K_root} output-layer factor trees ...')

fig, axes = plt.subplots(1, K_root, figsize=(6 * K_root, 4.5), squeeze=False)
axes = axes[0]

for k in range(K_root):
    acts, top_idx = top_stimuli_factor_activations(
        factor_nodes, root_bft_node, k, TOP_N_FACTOR
    )
    lam = root_bft_node['lambdas'][k]
    class_counts = {CLASS_NAMES[c]: int((all_targets[top_idx] == c).sum())
                    for c in sorted(np.unique(all_targets))}
    plot_factor_tree(
        factor_nodes, acts, ax=axes[k],
        title=f'Output factor {k}  λ={lam:.3f}\nTop {len(top_idx)}: {class_counts}',
    )

plt.suptitle('Factor trees — coloured by top stimuli of each OUTPUT (last-layer) factor',
             y=1.02, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'tree_output_factors.png'), dpi=120, bbox_inches='tight')
plt.show()

## Section 3: Factor Fingerprints & Stimulus Similarity

Each stimulus is represented as the concatenation of `img_factors[s, :]` across all tree
nodes (BFS order, all K factors per node). Pairwise cosine similarity reveals how well
the factor decomposition separates digit classes.

In [ ]:
F = extract_fingerprint_matrix(tree_root, np.arange(n_samples))
expected_dim = sum(len(e['lambdas']) for e in tree_nodes)
print(f'Fingerprint matrix: {F.shape}   (Σ K_i = {expected_dim})')
assert F.shape[1] == expected_dim

In [ ]:
MAX_VIZ = 500
rng = np.random.default_rng(42)
if n_samples > MAX_VIZ:
    # Stratified sample: keep proportional representation per digit class
    viz_idx = np.concatenate([
        rng.choice(np.where(all_targets == c)[0],
                   min(MAX_VIZ // 10, (all_targets == c).sum()), replace=False)
        for c in range(10)
    ])
    viz_idx = viz_idx[np.argsort(all_targets[viz_idx])]
else:
    viz_idx = np.argsort(all_targets)

S = compute_stimulus_similarity(F[viz_idx])
targets_viz = all_targets[viz_idx]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(S, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label='Cosine similarity')

# Digit-class boundaries
for cl in range(10):
    for b in np.where(np.diff((targets_viz == cl).astype(int)))[0]:
        ax.axhline(b + 0.5, color='k', lw=0.5, alpha=0.6)
        ax.axvline(b + 0.5, color='k', lw=0.5, alpha=0.6)

ax.set(title=f'Factor fingerprint cosine similarity  (N={len(viz_idx)}, sorted by digit class)',
       xlabel='Stimulus', ylabel='Stimulus')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'similarity_heatmap.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import cosine_distances
from sklearn.decomposition import PCA

dist_matrix = cosine_distances(F[viz_idx])
coords = MDS(n_components=2, dissimilarity='precomputed', random_state=0, n_init=4).fit_transform(dist_matrix)

# PCA over neurons — same stimuli as MDS
_li_sub  = np.hstack([l[viz_idx] for l in layer_inputs])
_li_last = layer_inputs[-2][viz_idx]
pca_all  = PCA(n_components=2).fit_transform(_li_sub)
pca_last = PCA(n_components=2).fit_transform(_li_last)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cmap_tab = plt.get_cmap('tab10')

for cl in range(10):
    mask = targets_viz == cl
    axes[0].scatter(coords[mask, 0], coords[mask, 1], c=[cmap_tab(cl)],
                    s=12, alpha=0.7, label=CLASS_NAMES[cl])
axes[0].legend(markerscale=2, title='Digit', ncol=2)
axes[0].set(title='MDS — factor fingerprint space (coloured by digit)', xlabel='MDS-1', ylabel='MDS-2')

for cl in range(10):
    mask = targets_viz == cl
    axes[1].scatter(pca_all[mask, 0], pca_all[mask, 1], c=[cmap_tab(cl)],
                    s=12, alpha=0.7, label=CLASS_NAMES[cl])
axes[1].legend(markerscale=2, title='Digit', ncol=2)
axes[1].set(title='PCA (all layers)', xlabel='PC1', ylabel='PC2')

for cl in range(10):
    mask = targets_viz == cl
    axes[2].scatter(pca_last[mask, 0], pca_last[mask, 1], c=[cmap_tab(cl)],
                    s=12, alpha=0.7, label=CLASS_NAMES[cl])
axes[2].legend(markerscale=2, title='Digit', ncol=2)
axes[2].set(title='PCA (last hidden layer)', xlabel='PC1', ylabel='PC2')

plt.suptitle('MDS vs PCA of network activations — test set (coloured by digit)', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'similarity_mds.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
S_all = compute_stimulus_similarity(F)
classes = sorted(np.unique(all_targets))
intra_vals, inter_vals = [], []
for ci, cl in enumerate(classes):
    mask = all_targets == cl
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for cl2 in classes[ci + 1:]:
        inter_vals.extend(S_all[np.ix_(mask, all_targets == cl2)].ravel())

intra_arr, inter_arr = np.array(intra_vals), np.array(inter_vals)
print(f'Intra-class similarity: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class similarity: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')

# Per-digit intra-class similarity
print('\nPer-digit intra-class similarity:')
for cl in classes:
    mask = all_targets == cl
    intra = S_all[np.ix_(mask, mask)]
    vals  = intra[np.triu_indices_from(intra, k=1)]
    print(f'  Digit {cl}: {vals.mean():.3f} ± {vals.std():.3f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(intra_arr, bins=60, alpha=0.6, label='Intra-class', density=True)
ax.hist(inter_arr, bins=60, alpha=0.6, label='Inter-class', density=True)
ax.axvline(intra_arr.mean(), color='C0', ls='--', lw=1.5)
ax.axvline(inter_arr.mean(), color='C1', ls='--', lw=1.5)
ax.set(xlabel='Cosine similarity', ylabel='Density',
       title='Factor fingerprint: intra- vs inter-class similarity (test set)')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'intra_inter_stats.png'), dpi=120, bbox_inches='tight')
plt.show()

## Section 4: ID Sanity Check

Project **training** samples (all 10 MNIST digits — same distribution as BFT) onto the
fixed factors via NNLS and compare their tree activation patterns to **test** samples.
If the factors are stable across train/test, patterns should be nearly identical.

In [ ]:
# Collect training layer inputs (correctly classified; cap at 500/class for speed)
data_train = collect_layer_inputs(
    model, train_loader.dataset,
    label_transform=label_transform, n_per_class=500, device=device,
)
train_images  = data_train['images']
train_targets = data_train['targets']
train_digits  = data_train['digits']
train_inputs  = data_train['layer_inputs']
print(f'Training samples: {len(train_images)}')
print(f'Class dist: {dict(zip(*np.unique(train_targets, return_counts=True)))}')

In [ ]:
projected_train_root  = project_stimuli_onto_tree(tree_root, train_inputs)
factor_nodes_train    = extract_factor_tree_nodes(projected_train_root)

In [ ]:
# Side-by-side tree per digit class: training vs test
n_classes = len(classes)
fig, axes = plt.subplots(2, n_classes, figsize=(4.5 * n_classes, 8))

for col, cl in enumerate(classes):
    cl_name = CLASS_NAMES[cl]

    train_cl_idx = np.where(train_targets == cl)[0]
    acts_train = compute_factor_activations(factor_nodes_train, train_cl_idx)
    plot_factor_tree(factor_nodes_train, acts_train, ax=axes[0, col],
                     title=f'TRAIN digit {cl_name}  (n={len(train_cl_idx)})')

    test_cl_idx = np.where(all_targets == cl)[0]
    acts_test = compute_factor_activations(factor_nodes, test_cl_idx)
    plot_factor_tree(factor_nodes, acts_test, ax=axes[1, col],
                     title=f'TEST digit {cl_name}  (n={len(test_cl_idx)})')

plt.suptitle('ID Sanity Check — Training vs Test factor activations per digit\n'
             '(patterns should match if factors are stable)', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'id_tree_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cross-similarity matrix: 3 representative digits (train + test)
N_PER_SPLIT = 100
rng = np.random.default_rng(0)
SHOW_DIGITS = [0, 1, 2, 3, 4]  # show first 5 digits for a readable heatmap

splits = {}
for cl in SHOW_DIGITS:
    cl_name = CLASS_NAMES[cl]
    tr_idx = np.where(train_targets == cl)[0]
    if len(tr_idx) > N_PER_SPLIT:
        tr_idx = rng.choice(tr_idx, N_PER_SPLIT, replace=False)
    splits[f'train_{cl_name}'] = extract_fingerprint_matrix(projected_train_root, tr_idx)

    te_idx = np.where(all_targets == cl)[0]
    if len(te_idx) > N_PER_SPLIT:
        te_idx = rng.choice(te_idx, N_PER_SPLIT, replace=False)
    splits[f'test_{cl_name}'] = extract_fingerprint_matrix(tree_root, te_idx)

split_labels = list(splits.keys())
F_blocks     = np.concatenate(list(splits.values()), axis=0)
block_sizes  = [len(v) for v in splits.values()]
block_starts = [0] + list(np.cumsum(block_sizes[:-1]))
block_ends   = list(np.cumsum(block_sizes))
boundaries   = np.array(block_ends[:-1])
S_cross      = compute_stimulus_similarity(F_blocks)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(S_cross, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label='Cosine similarity')

for b in boundaries:
    ax.axhline(b - 0.5, color='k', lw=1.5)
    ax.axvline(b - 0.5, color='k', lw=1.5)

centres = np.array(block_starts) + np.array(block_sizes) / 2
ax.set_xticks(centres); ax.set_xticklabels(split_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticks(centres); ax.set_yticklabels(split_labels, fontsize=8)
ax.set_title('ID Sanity Check — cross-similarity matrix (digits 0–4)\n'
             'Same-digit train/test blocks should be dark red (high similarity)')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'id_fingerprint_comparison.png'), dpi=120, bbox_inches='tight')
plt.show()

print('Mean cosine similarity per block pair:')
header = ''.join(f'{l:>16}' for l in split_labels)
print(f'{"":>16}{header}')
for i, li in enumerate(split_labels):
    row_vals = []
    for j, lj in enumerate(split_labels):
        block = S_cross[block_starts[i]:block_ends[i], block_starts[j]:block_ends[j]]
        vals  = block[np.triu_indices_from(block, k=1)] if i == j else block.ravel()
        row_vals.append(f'{vals.mean():>16.3f}')
    print(f'{li:>16}' + ''.join(row_vals))

## Section 5: Real OOD — Fashion-MNIST

The digit MLP was trained on **all 10 MNIST digits** — there are no excluded training digits
to serve as near-OOD.  Instead we use **Fashion-MNIST** (FashionMNIST): 10,000 test images
of clothing items, also 28×28 grayscale.  The model has never seen fashion items; they
represent a true **domain shift** (different textures, shapes, and statistics).

Fashion-MNIST classes (0–9): T-shirt/top, Trouser, Pullover, Dress, Coat,
Sandal, Shirt, Sneaker, Bag, Ankle boot.

In [ ]:
fmnist_test = datasets.FashionMNIST('../data/', train=False, download=True, transform=ToTensor())

data_ood = collect_layer_inputs_generic(
    model, fmnist_test,
    label_transform=label_transform,
    only_correct=False, device=device,
)

idx = np.random.choice(len(data_ood['images']), 1000)
ood_images  = data_ood['images'][idx]       # (N, 1, 28, 28)
ood_digits  = data_ood['digits'][idx]        # FMNIST class labels (0=T-shirt, ...)
ood_targets = data_ood['targets'][idx]      # same (identity transform)
ood_preds   = data_ood['preds'][idx]         # which MNIST digit the model predicts
ood_inputs  = [layer[idx] for layer in data_ood['layer_inputs']]

n_ood = len(ood_images)
# 'accuracy' here means: does the predicted digit class coincide with the FMNIST class number?
# (meaningless, reported only as a sanity check — should be ~10%)
ood_acc = (ood_preds == ood_targets).mean()
print(f'Fashion-MNIST samples   : {n_ood}')
print(f'Model accuracy on FMNIST: {ood_acc:.3f}  (chance = 0.1)')
print(f'FMNIST class breakdown  : {dict(zip(*np.unique(ood_digits, return_counts=True)))}')
print()
print('Digit predictions for each FMNIST class:')
for fc in range(10):
    mask = ood_digits == fc
    pred_dist = dict(zip(*np.unique(ood_preds[mask], return_counts=True)))
    print(f'  {FMNIST_CLASSES[fc]:15s}: {pred_dist}')

In [ ]:
projected_ood_root  = project_stimuli_onto_tree(tree_root, ood_inputs)
factor_nodes_ood    = extract_factor_tree_nodes(projected_ood_root)

In [ ]:
# ── Tree per Fashion-MNIST class ───────────────────────────────────────────────
unique_fmnist = sorted(np.unique(ood_digits))
ncols = min(5, len(unique_fmnist))
nrows = int(np.ceil(len(unique_fmnist) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)

for i, fc in enumerate(unique_fmnist):
    ax = axes[i // ncols][i % ncols]
    fc_idx = np.where(ood_digits == fc)[0]
    acts = compute_factor_activations(factor_nodes_ood, fc_idx)
    top_pred = np.bincount(ood_preds[fc_idx]).argmax()
    plot_factor_tree(
        factor_nodes_ood, acts, ax=ax,
        title=f'OOD: {FMNIST_CLASSES[fc]}  n={len(fc_idx)}\ntop digit pred={top_pred}',
    )

for j in range(len(unique_fmnist), nrows * ncols):
    axes[j // ncols][j % ncols].set_visible(False)

plt.suptitle('Real OOD — Factor tree activation per Fashion-MNIST class', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'ood_tree_per_fmnist_class.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── MDS + PCA: ID test vs Fashion-MNIST OOD in factor space ──────────────────
N_ID_SAMPLE  = 200
N_OOD_SAMPLE = min(200, n_ood)

rng = np.random.default_rng(1)
id_sub   = rng.choice(n_samples, N_ID_SAMPLE, replace=False)
ood_sub  = rng.choice(n_ood,     N_OOD_SAMPLE, replace=False)

F_id      = extract_fingerprint_matrix(tree_root, id_sub)
F_ood_sub = extract_fingerprint_matrix(projected_ood_root, ood_sub)
F_joint   = np.concatenate([F_id, F_ood_sub], axis=0)

coords = MDS(n_components=2, dissimilarity='precomputed', random_state=0, n_init=4).fit_transform(
    cosine_distances(F_joint)
)

# PCA over neurons for the same joint stimulus set
_li_id   = np.hstack([l[id_sub]  for l in layer_inputs])
_li_ood  = np.hstack([l[ood_sub] for l in ood_inputs])
_li_joint_all  = np.vstack([_li_id,  _li_ood])
_li_joint_last = np.vstack([layer_inputs[-2][id_sub], ood_inputs[-2][ood_sub]])
pca_all  = PCA(n_components=2).fit_transform(_li_joint_all)
pca_last = PCA(n_components=2).fit_transform(_li_joint_last)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
cmap_tab = plt.get_cmap('tab10')

# MDS: ID vs OOD colouring
axes[0].scatter(coords[:N_ID_SAMPLE, 0], coords[:N_ID_SAMPLE, 1],
                c='steelblue', s=10, alpha=0.6, label='ID test (MNIST digits)')
axes[0].scatter(coords[N_ID_SAMPLE:, 0], coords[N_ID_SAMPLE:, 1],
                c='tomato',    s=10, alpha=0.6, label='OOD (Fashion-MNIST)')
axes[0].legend(markerscale=3)
axes[0].set(title='MDS — MNIST digits vs Fashion-MNIST', xlabel='MDS-1', ylabel='MDS-2')

# MDS: colour by digit class (ID) and FMNIST class (OOD)
id_digits  = all_digits[id_sub]
ood_fclasses = ood_digits[ood_sub]
for d in sorted(np.unique(id_digits)):
    mask = id_digits == d
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    c=[cmap_tab(d)], marker='o', s=14, alpha=0.7, label=f'digit {d}')
for fc in sorted(np.unique(ood_fclasses)):
    mask_ood = ood_fclasses == fc
    idx = np.arange(N_ID_SAMPLE, N_ID_SAMPLE + N_OOD_SAMPLE)[mask_ood]
    axes[1].scatter(coords[idx, 0], coords[idx, 1],
                    c=[cmap_tab(fc)], marker='s', s=14, alpha=0.5,
                    label=f'FM:{FMNIST_CLASSES[fc][:6]}')
axes[1].legend(markerscale=2, ncol=2, fontsize=7, title='class (■=OOD FashionMNIST)')
axes[1].set(title='MDS — coloured by class  (○=MNIST digit, ■=FashionMNIST)', xlabel='MDS-1', ylabel='MDS-2')

# PCA — all layers, ID/OOD colouring
axes[2].scatter(pca_all[:N_ID_SAMPLE, 0], pca_all[:N_ID_SAMPLE, 1],
                c='steelblue', s=10, alpha=0.6, label='ID test (MNIST)')
axes[2].scatter(pca_all[N_ID_SAMPLE:, 0], pca_all[N_ID_SAMPLE:, 1],
                c='tomato',    s=10, alpha=0.6, label='OOD (FashionMNIST)')
axes[2].legend(markerscale=3)
axes[2].set(title='PCA (all layers) — ID vs OOD', xlabel='PC1', ylabel='PC2')

# PCA — last hidden layer, ID/OOD colouring
axes[3].scatter(pca_last[:N_ID_SAMPLE, 0], pca_last[:N_ID_SAMPLE, 1],
                c='steelblue', s=10, alpha=0.6, label='ID test (MNIST)')
axes[3].scatter(pca_last[N_ID_SAMPLE:, 0], pca_last[N_ID_SAMPLE:, 1],
                c='tomato',    s=10, alpha=0.6, label='OOD (FashionMNIST)')
axes[3].legend(markerscale=3)
axes[3].set(title='PCA (last hidden layer) — ID vs OOD', xlabel='PC1', ylabel='PC2')

plt.suptitle('OOD factor fingerprint space: MNIST digits vs Fashion-MNIST', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'ood_fingerprint_mds.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Block similarity: ID-digits vs OOD-FashionMNIST ──────────────────────────
N_BLOCK = 80
rng_block = np.random.default_rng(2)
BLOCK_DIGITS = [0, 1, 2, 4, 6]  # representative ID digits
BLOCK_FMNIST = [0, 1, 7, 9]     # T-shirt, Trouser, Sneaker, Ankle boot

blocks = {}
for cl in BLOCK_DIGITS:
    idx = rng_block.choice(np.where(all_targets == cl)[0],
                            min(N_BLOCK, (all_targets == cl).sum()), replace=False)
    blocks[f'ID-{cl}'] = extract_fingerprint_matrix(tree_root, idx)

for fc in BLOCK_FMNIST:
    idx = rng_block.choice(np.where(ood_digits == fc)[0],
                            min(N_BLOCK, (ood_digits == fc).sum()), replace=False)
    blocks[f'FM-{FMNIST_CLASSES[fc][:6]}'] = extract_fingerprint_matrix(projected_ood_root, idx)

block_labels  = list(blocks.keys())
F_all_blocks  = np.concatenate(list(blocks.values()), axis=0)
bl_sizes      = [len(v) for v in blocks.values()]
bl_starts     = [0] + list(np.cumsum(bl_sizes[:-1]))
bl_ends       = list(np.cumsum(bl_sizes))
bounds        = np.array(bl_ends[:-1])
S_blocks      = compute_stimulus_similarity(F_all_blocks)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(S_blocks, aspect='auto', interpolation='nearest', cmap='RdBu_r', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, label='Cosine similarity')
for b in bounds:
    ax.axhline(b - 0.5, color='k', lw=1.5)
    ax.axvline(b - 0.5, color='k', lw=1.5)
centres = np.array(bl_starts) + np.array(bl_sizes) / 2
ax.set_xticks(centres); ax.set_xticklabels(block_labels, rotation=35, ha='right')
ax.set_yticks(centres); ax.set_yticklabels(block_labels)
ax.set_title('ID digits vs OOD Fashion-MNIST fingerprint cross-similarity\n'
             'Low off-diagonal (ID ↔ OOD) = factor space is domain-specific')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'ood_cross_similarity.png'), dpi=120, bbox_inches='tight')
plt.show()

## Section 5b: Far OOD — Synthetic images with no digit structure

Four synthetic image types:
- **gaussian_noise** — i.i.d. pixels ~ N(0.5, 0.25²), clipped to [0, 1]
- **uniform_gray** — constant mid-grey (all pixels = 0.5)
- **checkerboard** — hard 0/1 alternating pattern at pixel scale
- **inverted_test** — 1 − (test digit image): same structure, opposite contrast

In [ ]:
N_FAR = 300
rng_f = np.random.default_rng(99)

_chk = (np.indices((IMAGE_SIDE, IMAGE_SIDE)).sum(axis=0) % 2).astype(np.float32)

far_ood_arrays = {
    'gaussian_noise': np.clip(
        rng_f.normal(0.5, 0.25, (N_FAR, 1, IMAGE_SIDE, IMAGE_SIDE)).astype(np.float32), 0, 1),
    'uniform_gray':   np.full((N_FAR, 1, IMAGE_SIDE, IMAGE_SIDE), 0.5, dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk, (N_FAR, 1, IMAGE_SIDE, IMAGE_SIDE)).copy().astype(np.float32),
    'inverted_test':  np.clip(1.0 - all_images[:N_FAR], 0, 1).astype(np.float32),
}

def collect_far_ood(model, images_np, label_transform, device, batch_size=256):
    dataset = TensorDataset(
        torch.from_numpy(images_np),
        torch.zeros(len(images_np), dtype=torch.long),
    )
    return collect_layer_inputs_generic(
        model, dataset, label_transform=label_transform,
        only_correct=False, device=device, batch_size=batch_size,
    )

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    d = collect_far_ood(model, imgs, label_transform, device)
    d['projected_root'] = project_stimuli_onto_tree(tree_root, d['layer_inputs'])
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name] = d
    acc = (d['preds'] == d['targets']).mean()
    print(f'{name:20s}  n={len(imgs)}  model acc={acc:.3f}')

In [ ]:
n_examples = 6
fig, axes = plt.subplots(len(far_ood_arrays), n_examples,
                          figsize=(n_examples * 1.5, len(far_ood_arrays) * 1.6))
for row, (name, imgs) in enumerate(far_ood_arrays.items()):
    for col in range(n_examples):
        axes[row, col].imshow(imgs[col, 0], cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=9, rotation=30, ha='right', va='center')
plt.suptitle('Far OOD — example images per type', y=1.01, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'far_ood_examples.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5), squeeze=False)
axes = axes[0]

for ax, (name, d) in zip(axes, far_ood_data.items()):
    all_idx = np.arange(len(d['images']))
    acts    = compute_factor_activations(d['factor_nodes'], all_idx)
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)

plt.suptitle('Far OOD — factor-level tree activation (all stimuli averaged per type)',
             y=1.02, fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'far_ood_factor_tree.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── MDS + PCA: ID test / Fashion-MNIST OOD / far OOD in factor fingerprint space ──
N_EACH_MDS = 80
rng_m = np.random.default_rng(7)

F_parts, li_all_parts, li_last_parts = [], [], []
mds_labels, mds_markers = [], []

# ID test
id_sub = rng_m.choice(n_samples, min(N_EACH_MDS, n_samples), replace=False)
F_parts.append(extract_fingerprint_matrix(tree_root, id_sub))
li_all_parts.append(np.hstack([l[id_sub] for l in layer_inputs]))
li_last_parts.append(layer_inputs[-2][id_sub])
mds_labels  += ['ID-MNIST'] * len(id_sub)
mds_markers += ['o'] * len(id_sub)

# Fashion-MNIST OOD
ood_sub = rng_m.choice(n_ood, min(N_EACH_MDS, n_ood), replace=False)
F_parts.append(extract_fingerprint_matrix(projected_ood_root, ood_sub))
li_all_parts.append(np.hstack([l[ood_sub] for l in ood_inputs]))
li_last_parts.append(ood_inputs[-2][ood_sub])
mds_labels  += ['OOD-FashionMNIST'] * len(ood_sub)
mds_markers += ['s'] * len(ood_sub)

# Far OOD types
marker_cycle = ['^', 'D', 'P', 'X']
for (name, d), mkr in zip(far_ood_data.items(), marker_cycle):
    n = min(N_EACH_MDS, len(d['images']))
    sub = rng_m.choice(len(d['images']), n, replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], sub))
    li_all_parts.append(np.hstack([l[sub] for l in d['layer_inputs']]))
    li_last_parts.append(d['layer_inputs'][-2][sub])
    mds_labels  += [name] * n
    mds_markers += [mkr] * n

F_joint    = np.concatenate(F_parts, axis=0)
coords_mds = MDS(n_components=2, dissimilarity='precomputed',
                 random_state=0, n_init=4).fit_transform(cosine_distances(F_joint))
pca_all  = PCA(n_components=2).fit_transform(np.vstack(li_all_parts))
pca_last = PCA(n_components=2).fit_transform(np.vstack(li_last_parts))

unique_labels = list(dict.fromkeys(mds_labels))
cmap_tab = plt.get_cmap('tab10')
label_color  = {l: cmap_tab(i % 10) for i, l in enumerate(unique_labels)}
label_marker = dict(zip(mds_labels, mds_markers))

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for embed, ax, title in [
    (coords_mds, axes[0], 'MDS — factor fingerprint space'),
    (pca_all,    axes[1], 'PCA (all layers)'),
    (pca_last,   axes[2], 'PCA (last hidden layer)'),
]:
    for i, lbl in enumerate(mds_labels):
        ax.scatter(embed[i, 0], embed[i, 1],
                   c=[label_color[lbl]], marker=label_marker[lbl], s=20, alpha=0.7)
    for lbl in unique_labels:
        ax.scatter([], [], c=[label_color[lbl]], marker=label_marker[lbl], s=50, label=lbl)
    ax.legend(markerscale=2, ncol=1, fontsize=9, title='stimulus type')
    ax.set(title=title, xlabel='Dim 1', ylabel='Dim 2')

plt.suptitle('ID MNIST / Fashion-MNIST OOD / far OOD — factor fingerprint MDS vs PCA of network activations',
             fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'far_ood_mds.png'), dpi=120, bbox_inches='tight')
plt.show()